In [1]:
import tempfile
import os

# 📍【关键修复 1】：重定向 Catalyst 的临时文件，彻底解决超算节点硬盘爆满的问题！
custom_tmp_path = os.path.join(os.getcwd(), "catalyst_tmp_cache")
os.makedirs(custom_tmp_path, exist_ok=True)
os.environ['TMPDIR'] = custom_tmp_path
tempfile.tempdir = custom_tmp_path
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import time
import math
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.sparse import coo_matrix

import jax
# 开启双精度
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import optax
import pennylane as qml
import catalyst
qml.about()

Name: pennylane
Version: 0.43.1
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages
Requires: appdirs, autograd, autoray, cachetools, diastatic-malt, networkx, numpy, packaging, pennylane-lightning, requests, rustworkx, scipy, tomlkit, typing_extensions
Required-by: pennylane_catalyst, pennylane_lightning, pennylane_lightning_gpu

Platform info:           Linux-6.11.0-26-generic-x86_64-with-glibc2.39
Python version:          3.12.12
Numpy version:           2.3.4
Scipy version:           1.16.2
JAX version:             0.6.2
Installed devices:
- default.clifford (pennylane-0.43.1)
- default.gaussian (pennylane-0.43.1)
- default.mixed (pennylane-0.43.1)
- default.qubit (pennylane-0.43.1)
- default.qutrit (pennylane-0.43.1)


In [2]:
# =================== 0. 环境与硬件检查 ===================
print('✅ JAX version:', jax.__version__)
print('✅ Devices:', jax.devices())
if any(d.platform == 'gpu' for d in jax.devices()):
    print('🎉 GPU is working!')
else:
    print('⚠️ No GPU detected')

✅ JAX version: 0.6.2
✅ Devices: [CudaDevice(id=0)]
🎉 GPU is working!


In [3]:
# =================== 1. 数据准备 (直接使用 A100 的大显存优势) ===================
matrix_file = "L=6 N=4.npz"
H_3 = sp.load_npz(matrix_file)

min_eigvals, _ = eigsh(H_3, k=1, which='SA')
exact_min_energy = float(min_eigvals[0])
print(f"最小特征值 (理论极限): {exact_min_energy:.8f}")

d = H_3.shape[0]
n_qubits = int(np.ceil(np.log2(d)))
l = 2 ** n_qubits
depth = math.ceil(2**n_qubits / n_qubits) + n_qubits
print(f'电路深度: {depth}, 参数总量: {depth * n_qubits}')

# 补全到 2^Nq 并 Gray 编码 (逻辑和之前一样)
H_3_coo = H_3.tocoo()
rows, cols, data = H_3_coo.row.astype(np.int64), H_3_coo.col.astype(np.int64), H_3_coo.data
if l > d:
    rows = np.concatenate([rows, np.arange(d, l, dtype=np.int64)])
    cols = np.concatenate([cols, np.arange(d, l, dtype=np.int64)])
    data = np.concatenate([data, np.full(l - d, 1000.0, dtype=data.dtype)])

def gray_code(n):
    if n == 1: return ["0", "1"]
    lower = gray_code(n - 1)
    return ["0" + x for x in lower] + ["1" + x for x in reversed(lower)]

gray_basis = gray_code(n_qubits)
gray2natural = np.array([int(g, 2) for g in gray_basis], dtype=np.int64)
new_rows, new_cols = gray2natural[rows], gray2natural[cols]
H_gray_csr = coo_matrix((data, (new_rows, new_cols)), shape=(l, l)).tocsr()

# 📍【关键修复 2】：A100 显存管够！直接将 4GB 的矩阵转化为 JAX 密集数组传入 GPU，速度拉满！
H_dense = H_gray_csr.toarray()
H_jax = jnp.array(H_dense, dtype=jnp.complex128)
del H_3, H_3_coo, H_gray_csr, H_dense

最小特征值 (理论极限): -28.08061951
电路深度: 1185, 参数总量: 16590


In [4]:
# =================== 2. 量子电路与 Catalyst 编译 ===================
hf = np.zeros(n_qubits, dtype=int)
dev = qml.device("lightning.gpu", wires=n_qubits)

# 📍【关键修改 1】：在 qnode 外部提前定义好一个纯 Python 列表！
all_wires = list(range(n_qubits))

# 📍【关键 2】：恢复 H_mat 参数
@qml.qnode(dev)
def cost_circuit(params_2d, H_mat):
    qml.BasisState(hf, wires=all_wires)

    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    for d_idx in range(depth):
        for i in range(n_qubits):
            qml.RY(params_2d[d_idx, i], wires=i)

        for i in range(0, n_qubits - 1, 2):
            qml.CNOT(wires=[i, i + 1])

        qml.CNOT(wires=[n_qubits-1, 0])

        for i in range(1, n_qubits - 1, 2):
            qml.CNOT(wires=[i, i + 1])

    return qml.expval(qml.Hermitian(H_mat, wires=all_wires))

# 初始化优化器
init_params = jnp.zeros((depth, n_qubits)) # 注意：这里改成了 2D 数组！
schedule = optax.exponential_decay(init_value=0.01, transition_steps=500, decay_rate=0.8)
opt = optax.adam(learning_rate=schedule)
opt_state = opt.init(init_params)

# 📍【关键 3】：恢复 H_matrix 的动态传参，防止撑爆 C++ 编译器！
@qml.qjit(autograph=True)
def update_step(current_params, current_opt_state, H_matrix):

    # 🌟 神级操作：在内部定义一个函数，只暴露 params 给求导器
    def cost_fn(p):
        return cost_circuit(p, H_matrix)

    # 此时无需写 argnums，因为 cost_fn 只有一个纯粹的参数 p
    energy, grads = catalyst.value_and_grad(cost_fn)(current_params)

    updates, next_opt_state = opt.update(grads, current_opt_state)
    next_params = optax.apply_updates(current_params, updates)

    return next_params, next_opt_state, energy

In [ ]:
# =================== 3. 极速训练循环 ===================
params = init_params
target_energy = exact_min_energy
tolerance = 1e-6
steps = 5000

print("\n🚀 开始 Catalyst + A100 狂暴模式训练...")
start_time = time.time()

for i in range(steps):
    # 📍【关键 4】：每次循环，把 H_jax 传给引擎 (显存中读取，极快！)
    params, opt_state, energy = update_step(params, opt_state, H_jax)

    current_energy = float(energy)

    if i % 50 == 0:
        print(f"Step = {i:4d},  Energy = {current_energy:.8f} Ha")

    if abs(current_energy - exact_min_energy) < tolerance:
        print(f"\n🎉 成功收敛！ 在第 {i} 步达到能量: {current_energy:.8f} Ha")
        break

print("-" * 30)
print(f"训练总耗时: {time.time() - start_time:.2f} 秒")
print(f"最终能量差: {abs(current_energy - exact_min_energy):.8e} Ha")



🚀 开始 Catalyst + A100 狂暴模式训练...
Step =    0,  Energy = 309.62899755 Ha
